In [1]:
# Cell 1: Imports and Setup
import chromadb
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from chromadb.api.types import EmbeddingFunction


c:\Users\suraj\LLM_projects\RAG_memory\LLM_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: Embedding Function Wrapper
class MiniLMEmbeddings(EmbeddingFunction):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def __call__(self, texts):
        return self.model.encode(texts).tolist()

    def name(self):
        return "all-MiniLM-L6-v2"

embedding_function = MiniLMEmbeddings()


In [3]:
# Cell 2: Embedding Function Wrapper
class MiniLMEmbeddings(EmbeddingFunction):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def __call__(self, texts):
        return self.model.encode(texts).tolist()

    def name(self):
        return "all-MiniLM-L6-v2"

embedding_function = MiniLMEmbeddings()


In [4]:
# Cell 3: ChromaDB Setup
client = chromadb.PersistentClient(path="./rag_db")

# Reset for clean run
try:
    client.delete_collection("rag_collection")
except:
    pass

collection = client.get_or_create_collection(
    name="rag_collection",
    embedding_function=embedding_function
)


In [5]:
# Cell 4: Load LLM
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)
llm2 = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Device set to use cuda:0


In [6]:
# Cell 5: Store a query-response pair in the vector DB
def store_rag_memory(query, response):
    document_text = f"Q: {query}\nA: {response}"
    doc_id = str(torch.randint(0, 1_000_000, (1,)).item())
    collection.add(documents=[document_text], ids=[doc_id])


In [7]:
def condense_memory(max_docs=100):
    """
    Summarize older query-response vectors if DB size exceeds max_docs.
    Compatible with recent ChromaDB versions.
    """
    # Get all documents and their metadata
    all_data = collection.get(include=["documents", "metadatas"])
    all_docs = all_data.get("documents", [])
    all_metadatas = all_data.get("metadatas", [])

    if len(all_docs) <= max_docs:
        return  # Nothing to condense

    print(f"⚠️ Memory exceeds {max_docs} entries — summarizing older entries...")

    # Take older half of documents for summarization
    half_index = len(all_docs) // 2
    docs_to_summarize = all_docs[:half_index]
    combined_text = "\n".join(docs_to_summarize)

    # Summarize using LLM2
    prompt = f"""
Summarize the following conversation history into a concise form preserving key Q&A for future reference:

{combined_text}

Summary:
"""
    summary = llm2(prompt, max_new_tokens=300, do_sample=True)[0]["generated_text"].strip()

    # Safely get IDs for deletion
    ids_to_delete = []
    for meta in all_metadatas[:half_index]:
        # Chroma stores IDs in metadata under "id"
        if meta is not None and "id" in meta:
            ids_to_delete.append(meta["id"])

    # Delete older documents
    if ids_to_delete:
        collection.delete(ids=ids_to_delete)

    # Add summarized document back with a new ID
    doc_id = str(torch.randint(0, 1_000_000, (1,)).item())
    collection.add(documents=[summary], ids=[doc_id])

    print("✅ Memory condensation complete.")


In [8]:
# Cell 7: RAG Query Function
def rag_query(question, k=3, store_response=True, max_docs=2):
    # 1️⃣ Search for top-k semantically similar past interactions
    results = collection.query(query_texts=[question], n_results=k)
    context = "\n".join(results["documents"][0]) if results["documents"][0] else ""

    # 2️⃣ Create prompt for the LLM
    prompt = f"""
Answer the question based largely on the context below. Treat the context below as a history of our past conversations. 
The last ones being more recent ones. If the answer is not in the context, say something related to it. 

Context:
{context}

Question: {question}
Answer:
    """

    answer = llm(prompt, max_new_tokens=150, do_sample=True)[0]["generated_text"].strip()

    # 3️⃣ Store the new query-response pair in the vector DB
    if store_response:
        store_rag_memory(question, answer)
        condense_memory(max_docs=max_docs)

    return answer


In [9]:
# Cell 8: Test Queries
if __name__ == "__main__":
    response1 = rag_query("What is vector database?")
    print("\nRAG Response 1:\n", response1)




RAG Response 1:
 Answer the question based largely on the context below. Treat the context below as a history of our past conversations. 
The last ones being more recent ones. If the answer is not in the context, say something related to it. 

Context:


Question: What is vector database?
Answer:
     A vector database is a type of database that stores data using vectors, which are mathematical representations of information such as text or images. These databases allow for efficient storage and retrieval of large amounts of structured and unstructured data by representing each piece of information as a point in a high-dimensional space. This approach enables fast search capabilities and can handle complex queries involving multiple pieces of data simultaneously.

In simpler terms, a vector database organizes and stores data points in a way that allows for quick access and analysis. Each "vector" represents an individual item of data (like a word from a document), and these vectors ar

In [10]:

response2 = rag_query("What is the size of your context memory?")
print("\nRAG Response 2:\n", response2)


RAG Response 2:
 Answer the question based largely on the context below. Treat the context below as a history of our past conversations. 
The last ones being more recent ones. If the answer is not in the context, say something related to it. 

Context:
Q: What is vector database?
A: Answer the question based largely on the context below. Treat the context below as a history of our past conversations. 
The last ones being more recent ones. If the answer is not in the context, say something related to it. 

Context:


Question: What is vector database?
Answer:
     A vector database is a type of database that stores data using vectors, which are mathematical representations of information such as text or images. These databases allow for efficient storage and retrieval of large amounts of structured and unstructured data by representing each piece of information as a point in a high-dimensional space. This approach enables fast search capabilities and can handle complex queries involvin

In [11]:
import matplotlib.pyplot as plt
import umap
import numpy as np

# 1️⃣ Get all embeddings and documents
all_data = collection.get(include=["documents", "embeddings"])
documents = all_data["documents"]
embeddings = np.array(all_data["embeddings"])

# 2️⃣ Reduce embeddings to 2D
reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, metric='cosine')
embeddings_2d = reducer.fit_transform(embeddings)

# 3️⃣ Plot
plt.figure(figsize=(12,8))
plt.scatter(embeddings_2d[:,0], embeddings_2d[:,1], s=50, alpha=0.7)

# Optional: label points with the first few characters of the doc
for i, doc in enumerate(documents):
    plt.text(embeddings_2d[i,0], embeddings_2d[i,1], doc[:15], fontsize=8)

plt.title("RAG Database Visualization (2D UMAP)")
plt.show()


Matplotlib is building the font cache; this may take a moment.


AttributeError: module 'umap' has no attribute 'UMAP'